# Clustering: GMM sweep and cluster anatomy

Unsupervised clustering of the frozen SimCLR and NNCLR features. Produces Table 4.8 and Figures 4.2–4.6 of the thesis.

For each domain and arm, the 512-d features are standardised and reduced to 50 principal components (both fit on the training split), and a Gaussian mixture with K components is fit on the training rows with no labels. Each cluster gets its training-set merger fraction, and a galaxy is scored by its soft cluster membership weighted by those fractions. Two modes: **flat** clusters the whole sample; **z-cond** clusters within redshift quartiles. Clustering is always in-domain. Horizon-AGN and HSC are clustered in the feature space of the TNG-trained encoder, so no cluster-to-label map is transferred.

Run order: sweep (Table 4.8, Fig 4.2) → K=8 gallery (Fig 4.4, saves the partition) → UMAP (Fig 4.3) → stage composition (Fig 4.5) → K=2 anatomy (Fig 4.6). The UMAP and composition cells read files saved by `results.ipynb` (`val_idx.npy`).

## Config

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

OUT = Path("/scratch/s4743091/tng/results_notebook/clustering"); OUT.mkdir(parents=True, exist_ok=True)
SPLIT_SEED, VAL_FRAC = 42, 0.2
K_GRID = [2, 4, 8, 16, 32, 64]
FAST_PASS = True
SEEDS  = [0, 1, 2] if FAST_PASS else [0, 1, 2, 3, 4]
N_INIT = 1
PCA_DIM = 50
Z_NBINS = 4

DOMAINS = {
  "tng": dict(
     labels="/scratch/s4743091/tng/processed_tng/labels_tng.npy",
     feats={"simclr":"/scratch/s4743091/tng/models/simclr_tng/latents_simclr.npy",
            "nnclr": "/scratch/s4743091/tng/models/nnclr_tng/latents_nnclr.npy"},
     has_seq=True, label_all=True),
  "horizon": dict(
     labels="/scratch/s4743091/horizon/labels_horizon.npy",
     feats={"simclr":"/scratch/s4743091/horizon/latents_simclr_horizon.npy",
            "nnclr": "/scratch/s4743091/horizon/latents_nnclr_horizon.npy"},
     has_seq=True, label_all=True),
  "hsc": dict(
     labels="/scratch/s4743091/hsc/labels_hsc.npy",
     feats={"simclr":"/scratch/s4743091/hsc/latents_simclr_hsc.npy",
            "nnclr": "/scratch/s4743091/hsc/latents_nnclr_hsc.npy"},
     has_seq=False, label_all=False),   # HSC: only visually labelled rows are scorable
}

## Per-domain loader

Merger-tree-grouped split where `sequence_ID` exists (TNG, Horizon-AGN), a plain split otherwise. Only labelled rows enter the cluster fractions and the scoring.

In [ ]:
def load_domain(dom):
    d = DOMAINS[dom]
    lab = np.load(d["labels"])
    y = lab["is_major_merger"].astype(int)
    z = lab["redshift"].astype(float)
    seq = lab["sequence_ID"] if d["has_seq"] else None
    labeled = np.ones(len(y), bool) if d["label_all"] else (lab["labeled"] == 1)
    idx = np.arange(len(y))
    if seq is not None:
        tr_i, va_i = next(GroupShuffleSplit(1, test_size=VAL_FRAC, random_state=SPLIT_SEED)
                          .split(idx, groups=seq))
    else:
        rng = np.random.default_rng(SPLIT_SEED); perm = rng.permutation(len(y))
        cut = int((1-VAL_FRAC)*len(y)); tr_i, va_i = perm[:cut], perm[cut:]
    tr = np.zeros(len(y), bool); tr[tr_i] = True
    va = np.zeros(len(y), bool); va[va_i] = True
    tr &= labeled; va &= labeled
    return dict(y=y, z=z, tr=tr, va=va, feats=d["feats"], n=len(y))

## Scorers

Both modes use soft scoring: a galaxy's score is its GMM membership probabilities multiplied by the per-cluster training merger fractions.

In [ ]:
def soft_flat(Xs, y, z, tr, va, K, seed):
    g = GaussianMixture(K, n_init=N_INIT, max_iter=200, random_state=seed).fit(Xs[tr])
    htr = g.predict(Xs[tr]); ytr = y[tr]
    f = np.array([ytr[htr==k].mean() if (htr==k).any() else ytr.mean() for k in range(K)])
    score = g.predict_proba(Xs) @ f
    return roc_auc_score(y[va], score[va])

def soft_zcond(Xs, y, z, tr, va, K, seed):
    edges = np.quantile(z[tr], np.linspace(0,1,Z_NBINS+1)); edges[0],edges[-1]=-np.inf,np.inf
    score = np.full(len(y), y[tr].mean(), float)
    for b in range(Z_NBINS):
        inb = (z>=edges[b]) & (z<edges[b+1])
        if (tr&inb).sum() < K*5: continue
        g = GaussianMixture(K, n_init=N_INIT, max_iter=200, random_state=seed).fit(Xs[tr&inb])
        htr = g.predict(Xs[tr&inb]); ytr_b = y[tr&inb]
        fb = np.array([ytr_b[htr==k].mean() if (htr==k).any() else ytr_b.mean() for k in range(K)])
        score[inb] = g.predict_proba(Xs[inb]) @ fb
    return roc_auc_score(y[va], score[va])

## Sweep

Resumable: each (domain, arm, K, mode) row is appended to the CSV as soon as it finishes, and finished rows are skipped on restart. The PCA-50 features are cached per (domain, arm) as `Xs_*.npy`; the figure cells below reuse them.

In [ ]:
from sklearn.decomposition import PCA
TABLE = OUT / (f"auc_sweep_pca{PCA_DIM}_" + ("fast" if FAST_PASS else "final") + ".csv")
COLS = ["domain","arm","K","mode","auc_mean","auc_std"]
if TABLE.exists():
    done_df = pd.read_csv(TABLE)
    done = set(zip(done_df.domain, done_df.arm, done_df.K.astype(int), done_df["mode"]))
    print(f"resuming: {len(done)} rows already done in {TABLE.name}")
else:
    pd.DataFrame(columns=COLS).to_csv(TABLE, index=False)
    done = set()
def append_row(r):
    pd.DataFrame([r], columns=COLS).to_csv(TABLE, mode="a", header=False, index=False)
def get_Xs(dom, arm, fpath, D):
    cache = OUT / f"Xs_{dom}_{arm}_pca{PCA_DIM}.npy"
    if cache.exists():
        return np.load(cache)
    X = np.load(fpath).astype(np.float32)
    assert len(X) == D["n"], f"{dom}/{arm}: {len(X)} feats vs {D['n']} labels"
    Xs = StandardScaler().fit(X[D["tr"]]).transform(X)
    pca = PCA(PCA_DIM, random_state=0).fit(Xs[D["tr"]])
    Xs = pca.transform(Xs)
    np.save(cache, Xs)
    print(f"  [{dom}/{arm}] PCA-{PCA_DIM} retains {pca.explained_variance_ratio_.sum():.1%} of h variance", flush=True)
    return Xs
for dom in DOMAINS:
    D = load_domain(dom)
    print(f"\n=== {dom}  n={D['n']:,}  tr={D['tr'].sum():,} va={D['va'].sum():,} "
          f"merger_frac={D['y'][D['tr']].mean():.3f} ===", flush=True)
    for arm, fpath in D["feats"].items():
        Xs = None
        for K in K_GRID:
            for mode, fn in [("flat", soft_flat), ("zcond", soft_zcond)]:
                key = (dom, arm, int(K), mode)
                if key in done:
                    print(f"  {arm:6s} K={K:3d} {mode:5s}  [cached]", flush=True)
                    continue
                if Xs is None:
                    Xs = get_Xs(dom, arm, fpath, D)
                aucs = [fn(Xs, D["y"], D["z"], D["tr"], D["va"], K, s) for s in SEEDS]
                r = dict(domain=dom, arm=arm, K=int(K), mode=mode,
                         auc_mean=float(np.mean(aucs)), auc_std=float(np.std(aucs)))
                append_row(r); done.add(key)
                print(f"  {arm:6s} K={K:3d} {mode:5s}  {r['auc_mean']:.3f} ± {r['auc_std']:.3f}", flush=True)
sweep = pd.read_csv(TABLE).sort_values(["domain","arm","K","mode"]).reset_index(drop=True)
print(f"\ntotal rows: {len(sweep)}")
sweep

## Table 4.8: clustering AUC at K = 64

*Added when rebuilding the repo: a pivot of the sweep table above.*

In [ ]:
(sweep[(sweep.K == 64) & sweep.domain.isin(["tng", "horizon"])]
 .pivot_table(index=["domain", "arm"], columns="mode", values="auc_mean").round(3))

## Figure 4.2: clustering AUC vs K

HSC z-cond is left out: with few labelled galaxies per redshift bin it is unreliable.

In [ ]:
STYLE = {("simclr","flat"):  ("#d1495b","--","SimCLR flat"),
         ("simclr","zcond"): ("#d1495b","-", "SimCLR z-cond"),
         ("nnclr","flat"):   ("#2e7d9c","--","NNCLR flat"),
         ("nnclr","zcond"):  ("#2e7d9c","-", "NNCLR z-cond")}
DOM_TITLE = {"tng":"TNG (in-domain)", "horizon":"Horizon-AGN", "hsc":"HSC (visual labels)"}
doms = [d for d in DOMAINS if (sweep.domain == d).any()]

fig, axes = plt.subplots(1, len(doms), figsize=(4.6*len(doms), 4.2), sharey=True, squeeze=False)
for a, dom in zip(axes[0], doms):
    sub = sweep[sweep.domain == dom]
    for arm in ["simclr","nnclr"]:
        for mode in ["flat","zcond"]:
            if dom == "hsc" and mode == "zcond":
                continue
            s = sub[(sub.arm==arm) & (sub["mode"]==mode)].sort_values("K")
            if s.empty: continue
            col, lsty, lab = STYLE[(arm,mode)]
            xpos = [K_GRID.index(int(k)) for k in s.K]
            a.errorbar(xpos, s.auc_mean, yerr=s.auc_std, ls=lsty, color=col, marker="o",
                       ms=5, lw=2, capsize=3, label=lab)
    a.set_xticks(range(len(K_GRID))); a.set_xticklabels(K_GRID)
    a.axhline(0.5, color="0.6", ls=":", lw=1)
    a.set_xlabel("$K$ (GMM components)"); a.set_title(DOM_TITLE.get(dom,dom))
    a.grid(axis="y", ls=":", lw=0.5, color="0.85"); a.spines[["top","right"]].set_visible(False)
axes[0][0].set_ylabel("Validation merger AUC"); axes[0][0].legend(fontsize=8, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig("auc_vs_K.pdf", bbox_inches="tight"); plt.show()

## Figure 4.4: K = 8 cluster gallery

Fits the flat K=8 partition on the sweep's training split (same grouped split, so it is the partition behind the sweep numbers), saves it, and shows random validation members of each cluster, sorted by merger fraction. Set `ARM` / `K` at the top; the first run fits and saves, later runs load.

In [ ]:
# ---------- pick these ----------
ARM, K, N_PER = "simclr", 8, 8
# --------------------------------

B = "/scratch/s4743091/tng/results_notebook/v3/"
P = "/scratch/s4743091/tng/processed_tng/"
C = Path("/scratch/s4743091/tng/results_notebook/clustering")

lab     = np.load(P + "labels_tng_seqsplit.npy", allow_pickle=True)
val_idx = np.load(B + "val_idx.npy")
y_full  = lab["is_major_merger"].astype(int)
seq     = lab["sequence_ID"]
images  = np.load(P + "images_tng_64.npy", mmap_mode="r")
Xs      = np.load(C / f"Xs_tng_{ARM}_pca50.npy")

idx = np.arange(len(y_full))
tr_i, _ = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(idx, groups=seq))
tr = np.zeros(len(y_full), bool); tr[tr_i] = True

fpart = C / f"clus_tng_{ARM}_K{K}.npy"
if fpart.exists():
    clus_full = np.load(fpart)
    print(f"loaded {fpart.name}")
else:
    clus_full = GaussianMixture(K, n_init=1, max_iter=200, random_state=0).fit(Xs[tr]).predict(Xs)
    np.save(fpart, clus_full); print(f"saved {fpart.name}")

clus = clus_full[val_idx]; yv = y_full[val_idx]
print(f"{ARM} K={K} | cluster sizes (val): {np.bincount(clus, minlength=K)}")

def show(ax, full_row):
    im = np.asarray(images[full_row], float)
    lo, hi = np.percentile(im, [0.5, 99.5])
    im = np.clip((im - lo) / (hi - lo + 1e-9), 0, 1)
    im = np.arcsinh(im / 0.1) / np.arcsinh(1 / 0.1)
    ax.imshow(im, origin="lower", cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])

mfrac = np.array([yv[clus == k].mean() if (clus == k).any() else 0 for k in range(K)])
order = np.argsort(-mfrac)
rng   = np.random.default_rng(1)

fig, axes = plt.subplots(K, N_PER, figsize=(N_PER * 1.3, K * 1.3), squeeze=False)
for r, k in enumerate(order):
    pool = np.where(clus == k)[0]
    pick = rng.choice(pool, min(N_PER, len(pool)), replace=False)
    for c in range(N_PER):
        if c < len(pick): show(axes[r, c], val_idx[pick[c]])
        else: axes[r, c].axis("off")
    axes[r, 0].set_ylabel(f"c{k}\nf={mfrac[k]:.2f}\nn={len(pool)}",
                          fontsize=7, rotation=0, labelpad=22, va="center")
fig.suptitle(f"{ARM.upper()} clusters (K={K}, flat) — rows sorted by merger fraction $f$", fontsize=11)
fig.tight_layout(); fig.savefig(f"cluster_gallery_{ARM}_K{K}.pdf", bbox_inches="tight"); plt.show()

## Figure 4.3: UMAP of the SimCLR representation

Embeds the held-out TNG set (the same 93,347 galaxies as the probe and the K=8 partition above).

*Rebuilt for the repo: the UMAP settings match the original results notebook (PCA-50, `n_neighbors=30`, `min_dist=0.1`); the loader lines that define `umap, y, z, stage, clus` were reconstructed. The plotting code is the original.*

In [ ]:
import umap as umap_lib
from sklearn.decomposition import PCA

fu = Path(B) / f"umap_val_{ARM}.npy"
if fu.exists():
    umap = np.load(fu)
else:
    Xv = np.load(DOMAINS["tng"]["feats"][ARM]).astype(np.float32)[val_idx]
    Xv = PCA(50, random_state=0).fit_transform(Xv)
    umap = umap_lib.UMAP(n_neighbors=30, min_dist=0.1, random_state=0).fit_transform(Xv)
    np.save(fu, umap)

lab_v = lab[val_idx]
y = lab_v["is_major_merger"].astype(int)
z = lab_v["redshift"].astype(float)
stage = np.select([lab_v["is_pre_merger"]==1, lab_v["is_ongoing_merger"]==1,
                   lab_v["is_post_merger"]==1], [1, 2, 3], default=0)
clus = np.load(C / f"clus_tng_{ARM}_K8.npy")[val_idx]
N = len(val_idx)
assert len(umap) == N

In [ ]:
rng = np.random.default_rng(0); sub = rng.choice(N, min(30000, N), replace=False); u = umap[sub]
fig, ax = plt.subplots(2, 2, figsize=(11, 10))

for v, c, l in [(0,"#cccccc","non-merger"), (1,"#d1495b","merger")]:
    m = y[sub]==v; ax[0,0].scatter(u[m,0], u[m,1], s=3, c=c, label=l, alpha=0.5, lw=0)
ax[0,0].legend(markerscale=4, fontsize=8, frameon=False); ax[0,0].set_title("merger label")

sc = ax[0,1].scatter(u[:,0], u[:,1], s=3, c=z[sub], cmap="viridis", alpha=0.6, lw=0)
plt.colorbar(sc, ax=ax[0,1], shrink=0.8, label="redshift $z$"); ax[0,1].set_title("redshift")

st = stage[sub]
ax[1,0].scatter(u[st==0,0], u[st==0,1], s=2, c="#dddddd", alpha=0.4, lw=0)
for v, c, l in [(1,"#4c9f70","pre"), (2,"#d1495b","ongoing"), (3,"#8a4fff","post")]:
    m = st==v; ax[1,0].scatter(u[m,0], u[m,1], s=4, c=c, label=l, alpha=0.7, lw=0)
ax[1,0].legend(markerscale=3, fontsize=8, frameon=False); ax[1,0].set_title("merger stage")

K = len(np.unique(clus))
ax[1,1].scatter(u[:,0], u[:,1], s=3, c=clus[sub], cmap="tab20", alpha=0.7, lw=0)
ax[1,1].set_title(f"GMM clusters (K={K})")

for a in ax.ravel(): a.set_xticks([]); a.set_yticks([]); a.set_aspect("equal")
fig.suptitle(f"UMAP of the {ARM.upper()} representation (held-out TNG)", fontsize=13)
fig.tight_layout(); fig.savefig(f"umap_{ARM}_panels.pdf", bbox_inches="tight"); plt.show()

## Figure 4.5: stage composition of the K = 8 clusters

In [ ]:
K, ARM = 8, "simclr"
clus = np.load(C / f"clus_tng_{ARM}_K{K}.npy")[val_idx]
yv, stv = y_full[val_idx], stage

comp  = np.array([[ (stv[clus==k]==s).mean() for s in range(4) ] for k in range(K)])
mfrac = np.array([ yv[clus==k].mean() for k in range(K) ]); o = np.argsort(-mfrac)
comp, sizes = comp[o], np.bincount(clus, minlength=K)[o]
names = ["non-merger","pre","ongoing","post"]; cols = ["#dddddd","#4c9f70","#d1495b","#8a4fff"]

fig, ax = plt.subplots(figsize=(7, 4)); bottom = np.zeros(K)
for s in range(4):
    ax.bar(range(K), comp[:,s], bottom=bottom, color=cols[s], label=names[s], width=0.9)
    bottom += comp[:,s]
ax.set_xticks(range(K)); ax.set_xticklabels([f"c{k}\nn={n}" for k,n in zip(o,sizes)], fontsize=7)
ax.set_ylim(0,1); ax.set_ylabel("fraction"); ax.set_xlabel("cluster (sorted by merger fraction)")
ax.legend(ncol=4, fontsize=8, loc="lower center", bbox_to_anchor=(0.5,1.02), frameon=False)
ax.spines[["top","right"]].set_visible(False)
fig.tight_layout(); fig.savefig(f"cluster_composition_{ARM}_K{K}.pdf", bbox_inches="tight"); plt.show()

## Figure 4.6: the K = 2 split

The representation's single binary division: flat K=2 on the training split, clusters ordered so cluster 0 is the merger-rich half. The table is computed on training rows; the gallery draws random members of each cluster.

*Adapted for the repo: the thesis figure shows three rows per cluster (the cell originally drew one), and the file name matches the thesis (`cluster_k2_examples.pdf`).*

In [ ]:
ARM = "simclr"
C = "/scratch/s4743091/tng/results_notebook/clustering/"
P = "/scratch/s4743091/tng/processed_tng/"

lab = np.load(P + "labels_tng.npy")
y   = lab["is_major_merger"].astype(int); z = lab["redshift"].astype(float)
seq = lab["sequence_ID"]
stage = np.select([lab["is_pre_merger"]==1, lab["is_ongoing_merger"]==1,
                   lab["is_post_merger"]==1], [1,2,3], default=0)
Xs  = np.load(C + f"Xs_tng_{ARM}_pca50.npy")
img = np.load(P + "images_tng_64.npy", mmap_mode="r")

tr_i,_ = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(np.arange(len(y)), groups=seq))
tr = np.zeros(len(y), bool); tr[tr_i] = True

g = GaussianMixture(2, n_init=1, max_iter=200, random_state=0).fit(Xs[tr])
clus = g.predict(Xs)

order = np.argsort([-y[(clus==k)&tr].mean() for k in range(2)])
remap = {old:new for new,old in enumerate(order)}
clus = np.array([remap[c] for c in clus])

print(f"{ARM} flat K=2 — characterisation (train-fraction based)\n")
print(f"{'cluster':>8}{'n':>9}{'merger f':>10}{'med z':>8}"
      f"{'%pre':>7}{'%ong':>7}{'%post':>7}")
for k in [0,1]:
    m = (clus==k)&tr
    print(f"{k:>8}{m.sum():>9}{y[m].mean():>10.3f}{np.median(z[m]):>8.2f}"
          f"{100*(stage[m]==1).mean():>7.1f}{100*(stage[m]==2).mean():>7.1f}"
          f"{100*(stage[m]==3).mean():>7.1f}")

def show(ax, r):
    im = np.asarray(img[r], float); lo,hi = np.percentile(im,[0.5,99.5])
    im = np.clip((im-lo)/(hi-lo+1e-9),0,1); im = np.arcsinh(im/0.1)/np.arcsinh(1/0.1)
    ax.imshow(im, origin="lower", cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])

rng = np.random.default_rng(1); npix, nrow = 10, 3
fig, axes = plt.subplots(2*nrow, npix, figsize=(npix*1.3, 2*nrow*1.4), squeeze=False)
for k in [0,1]:
    pool = np.where(clus==k)[0]; pick = rng.choice(pool, npix*nrow, replace=False)
    for r in range(nrow):
        for c in range(npix): show(axes[k*nrow + r, c], pick[r*npix + c])
    axes[k*nrow + 1, 0].set_ylabel(f"c{k}\nf={y[(clus==k)&tr].mean():.2f}\nmed z={np.median(z[clus==k]):.2f}",
                                   fontsize=7, rotation=0, labelpad=24, va="center")
fig.suptitle(f"{ARM.upper()} K=2 split — the representation's single binary division", fontsize=11)
fig.tight_layout(); fig.savefig("cluster_k2_examples.pdf", bbox_inches="tight"); plt.show()